<a href="https://colab.research.google.com/github/Usubillaga/Primum/blob/gh-pages/Stocks_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === ANÁLISIS FUNDAMENTAL DE ACCIONES CON DCF EN GOOGLE COLAB ===
# Ejecuta esta celda completa (Ctrl + Enter)

!pip install yfinance --quiet  # Instala yfinance si no está

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# ETFs por sector para comparación
SECTOR_ETFS = {
    'Technology': 'XLK',
    'Healthcare': 'XLV',
    'Financial Services': 'XLF',
    'Consumer Cyclical': 'XLY',
    'Communication Services': 'XLC',
    'Industrials': 'XLI',
    'Consumer Defensive': 'XLP',
    'Energy': 'XLE',
    'Utilities': 'XLU',
    'Real Estate': 'XLRE',
    'Basic Materials': 'XLB'
}

def analizar_accion(ticker):
    print(f"🔍 Analizando {ticker.upper()}...\n")

    stock = yf.Ticker(ticker)
    info = stock.info

    # Datos básicos
    nombre = info.get('longName', ticker)
    sector = info.get('sector', 'No disponible')
    precio_actual = info.get('currentPrice') or info.get('regularMarketPrice', 'N/A')

    print(f"🏢 Empresa: {nombre}")
    print(f"📊 Sector: {sector}")
    print(f"💰 Precio actual: ${precio_actual:.2f}\n" if precio_actual != 'N/A' else "💰 Precio actual: No disponible\n")

    # Métricas clave
    print("📈 MÉTRICAS FINANCIERAS CLAVE:")

    metrics = {
        'Margen Bruto': info.get('grossMargins'),
        'Margen Neto': info.get('profitMargins'),
        'ROE (Retorno sobre Equity)': info.get('returnOnEquity'),
        'ROA (Retorno sobre Activos)': info.get('returnOnAssets'),
        'P/E Ratio': info.get('trailingPE'),
        'P/B Ratio': info.get('priceToBook'),
        'EV/EBITDA': info.get('enterpriseToEbitda'),
        'Deuda / Patrimonio': info.get('debtToEquity'),
    }

    etf = SECTOR_ETFS.get(sector)
    if etf:
        sector_stock = yf.Ticker(etf)
        sector_info = sector_stock.info

    for nombre_met, valor in metrics.items():
        if valor is not None:
            if 'Margen' in nombre_met or 'RO' in nombre_met:
                valor_form = f"{valor*100:.2f}%"
            elif isinstance(valor, (int, float)):
                valor_form = f"{valor:.2f}"
            else:
                valor_form = valor

            # Comparación con sector si disponible
            sector_val = 'N/A'
            if etf:
                key_map = {
                    'Margen Bruto': 'grossMargins',
                    'Margen Neto': 'profitMargins',
                    'ROE': 'returnOnEquity',
                    'ROA': 'returnOnAssets',
                    'P/E Ratio': 'trailingPE',
                    'P/B Ratio': 'priceToBook',
                    'EV/EBITDA': 'enterpriseToEbitda'
                }
                key = key_map.get(nombre_met)
                if key:
                    s_val = sector_info.get(key)
                    if s_val is not None:
                        if 'Margen' in nombre_met or 'RO' in nombre_met:
                            sector_val = f"{s_val*100:.2f}%"
                        else:
                            sector_val = f"{s_val:.2f}"

            print(f"   • {nombre_met}: {valor_form}  (Sector: {sector_val})")
        else:
            print(f"   • {nombre_met}: No disponible")

    # === Cálculo simple de WACC ===
    print("\n💸 CÁLCULO DE WACC (Coste Medio Ponderado del Capital):")
    beta = info.get('beta', 1.0)
    try:
        rf = yf.Ticker("^TNX").history(period="1d")['Close'].iloc[-1] / 100
    except:
        rf = 0.043  # ~4.3% actual (bono 10 años EE.UU.)

    erp = 0.05  # Prima de riesgo mercado
    coste_equity = rf + beta * erp

    deuda_total = info.get('totalDebt', 0)
    market_cap = info.get('marketCap', 0)
    if market_cap > 0:
        peso_equity = market_cap / (market_cap + deuda_total) if (market_cap + deuda_total) > 0 else 1
        peso_deuda = deuda_total / (market_cap + deuda_total) if (market_cap + deuda_total) > 0 else 0
        wacc = peso_equity * coste_equity + peso_deuda * 0.05 * (1 - 0.21)  # coste deuda ~5%, impuesto 21%
        print(f"   • WACC estimado: {wacc*100:.2f}%")
    else:
        wacc = 0.10
        print(f"   • WACC estimado (por defecto): 10.00%")

    # === DCF BÁSICO ===
    print("\n📊 VALORACIÓN POR DCF (Flujos de Caja Descontados):")

    cf = stock.cashflow
    if not cf.empty:
        # Intentar obtener Free Cash Flow
        if 'Free Cash Flow' in cf.index:
            fcf_hist = cf.loc['Free Cash Flow'].dropna()
        else:
            ocf = cf.loc['Operating Cash Flow'] if 'Operating Cash Flow' in cf.index else pd.Series()
            capex = cf.loc['Capital Expenditures'] if 'Capital Expenditures' in cf.index else pd.Series()
            fcf_hist = ocf + capex  # CapEx es negativo

        if not fcf_hist.empty and fcf_hist.iloc[0] > 0:
            fcf_promedio = fcf_hist.iloc[:3].mean()

            # Inputs del usuario
            print("\nIngresa tus estimaciones:")
            crecimiento = float(input("   • Tasa de crecimiento anual esperada (ej. 10 para 10%): ") or 10) / 100
            crecimiento_perpetuo = float(input("   • Crecimiento perpetuo a largo plazo (ej. 2 para 2%): ") or 2) / 100
            años_proyeccion = 5

            # Proyecciones
            fcfs = [fcf_promedio * (1 + crecimiento)**(i+1) for i in range(años_proyeccion)]
            valor_terminal = fcfs[-1] * (1 + crecimiento_perpetuo) / (wacc - crecimiento_perpetuo)
            flujos_descontados = [f / (1 + wacc)**(i+1) for i, f in enumerate(fcfs)]
            vt_descontado = valor_terminal / (1 + wacc)**años_proyeccion

            valor_empresa = sum(flujos_descontados) + vt_descontado
            deuda_neta = info.get('totalDebt', 0) - info.get('cash', 0)
            valor_equity = max(valor_empresa - deuda_neta, 0)
            acciones = info.get('sharesOutstanding', 1)
            valor_intrinseco = valor_equity / acciones

            print(f"\n🎯 VALOR INTRÍNSECO POR ACCIÓN (DCF): ${valor_intrinseco:.2f}")
            print(f"💵 Precio actual: ${precio_actual:.2f}" if precio_actual != 'N/A' else "💵 Precio actual: No disponible")

            if precio_actual != 'N/A':
                if valor_intrinseco > precio_actual * 1.2:
                    print("🟢 CONCLUSIÓN: POSIBLEMENTE SUBVALORADA")
                elif valor_intrinseco < precio_actual * 0.8:
                    print("🔴 CONCLUSIÓN: POSIBLEMENTE SOBREVALORADA")
                else:
                    print("🟡 CONCLUSIÓN: PRECIO RAZONABLE")
        else:
            print("   No se pudo calcular DCF: FCF no disponible o negativo.")
    else:
        print("   No se pudo obtener datos de flujo de caja.")

# === EJECUTAR EL ANÁLISIS ===
print("🚀 ANALIZADOR DE ACCIONES CON DCF - GOOGLE COLAB")
print(f"Fecha: {datetime.now().strftime('%d/%m/%Y')}\n")

ticker_input = input("Ingresa el ticker de la acción (ej. AAPL, TSLA, MSFT): ").strip().upper()

if ticker_input:
    analizar_accion(ticker_input)
else:
    print("No se ingresó ningún ticker.")

🚀 ANALIZADOR DE ACCIONES CON DCF - GOOGLE COLAB
Fecha: 29/12/2025

